# Funnel system, cross-validated over the WHOLE dataset

Every article goes through RoBERTa; the AI pile trains and is judged by LIFE, the human pile
by BERT; one accuracy over **all 20,505 GossipCop / 520 PolitiFact articles**.

Coverage comes from **5-fold cross-validation**: the dataset is dealt into 5 folds over id-groups,
and round k trains on the other four and predicts fold k. Concatenated, every article has exactly
one prediction from models that never trained on it, so the pooled row is honest.

**RoBERTa filters training too.** Fold k's router predicts fold k, so merging the five prediction
files (`build_router_oof.py`) gives every article a router label from a model that did not see it.
Each expert's training pile is built from those labels: LIFE gets everything called AI (misrouted
human articles included), BERT everything called human. Veracity labels stay ground truth.

**LIFE's method is unchanged** - same LLaMA2-7B reconstruction features, same key-sentence top-k
(10 PolitiFact / 15 GossipCop), same sigmoid+BCE head (Eq 11-12). Only the population it is
fitted on changes.

| variant | routing | machine branch | reads |
|---|---|---|---|
| `funnel` | RoBERTa | LIFE | the system as specified |
| `oracle` | true provenance | LIFE | funnel - oracle = cost of router errors |
| `two_bert` | RoBERTa | plain BERT | funnel - two_bert = LIFE's contribution |
| `oracle_two_bert` | true provenance | plain BERT | both above, jointly |
| `monolithic` | none | - | one BERT on everything: did routing help at all |

Order matters: **routers for all 5 folds must finish before any expert cell**, because the
experts' training piles come from the merged router labels. Every cell is resumable
(`--skip_if_exists`), so a disconnect costs at most the fold in progress.

The 70/15/15 held-out run lives in `funnel_colab.ipynb` and is untouched by this notebook
(separate `run_*_cv/` directories).

In [ ]:
# GPU + deps (transformers, fastNLP etc. for the LIFE branch)
!nvidia-smi
%pip install -q -r requirements.txt

Wed Jul 29 10:16:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/LIFE'

DATASET_ROOT = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset'
POLITIFACT_DIR = f'{DATASET_ROOT}/PolitiFact++'
GOSSIPCOP_DIR = f'{DATASET_ROOT}/GossipCop++'

# LIFE features already on Drive (LLaMA2-7B reconstruction, all four subsets)
PF_FEATURES = f'{PROJECT_DIR}/dataset/features_llama_multi'
GC_FEATURES = f'{PROJECT_DIR}/dataset/gossipcop/features_llama_multi'

# Cross-validated run dirs - separate from the 70/15/15 run
PF_RUN = f'{PROJECT_DIR}/funnel_system/run_politifact_cv'
GC_RUN = f'{PROJECT_DIR}/funnel_system/run_gossipcop_cv'
for d in (PF_RUN, GC_RUN):
    os.makedirs(d, exist_ok=True)

os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())
print('PF features:', os.path.isdir(PF_FEATURES), '| GC features:', os.path.isdir(GC_FEATURES))

Mounted at /content/drive
cwd: /content/drive/MyDrive/LIFE
PF features: True | GC features: True


## PolitiFact++ - 520 articles, 5 folds

Runs in minutes; treat it as the plumbing check for the GossipCop run below, not as a result
(~104 articles per fold).

In [ ]:
!python funnel_system/make_folds.py --data_dir "{POLITIFACT_DIR}" --out_dir "{PF_RUN}" --k 5 --seed 7 --name PolitiFact++

[PolitiFact++] 520 articles in 290 id-groups -> 5 folds (assignment seed 7)

fold 0 -> /content/drive/MyDrive/LIFE/funnel_system/run_politifact_cv/fold0.json
[PolitiFact++] fold 0: articles per subset x split:
  HF: train=    68 val=     8 test=    21
  HR: train=   139 val=    15 test=    40
  MF: train=    68 val=     8 test=    21
  MR: train=    93 val=    11 test=    28
  test: 110 articles | fake 42 (38.2%) | AI 49 (44.5%)

fold 1 -> /content/drive/MyDrive/LIFE/funnel_system/run_politifact_cv/fold1.json
[PolitiFact++] fold 1: articles per subset x split:
  HF: train=    70 val=     8 test=    19
  HR: train=   140 val=    15 test=    39
  MF: train=    70 val=     8 test=    19
  MR: train=    92 val=    14 test=    26
  test: 103 articles | fake 38 (36.9%) | AI 45 (43.7%)

fold 2 -> /content/drive/MyDrive/LIFE/funnel_system/run_politifact_cv/fold2.json
[PolitiFact++] fold 2: articles per subset x split:
  HF: train=    70 val=     8 test=    19
  HR: train=   140 val=    15 test

In [ ]:
# Routers first: fold k's router predicts fold k, which is what makes the merged
# labels out-of-fold. Nothing downstream can run until all 5 are written.
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold0.json" --role router --name PolitiFact++ --out_dir "{PF_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold1.json" --role router --name PolitiFact++ --out_dir "{PF_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold2.json" --role router --name PolitiFact++ --out_dir "{PF_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold3.json" --role router --name PolitiFact++ --out_dir "{PF_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold4.json" --role router --name PolitiFact++ --out_dir "{PF_RUN}" --skip_if_exists

[PolitiFact++ | router | seed=0] train=358 val=42 test=110 (full shared test) | model=roberta-base label=provenance device=cuda
config.json: 100% 481/481 [00:00<00:00, 2.31MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 128kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 21.4MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 33.6MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 27.7MB/s]

model.safetensors: downloading bytes:  29% 143M/499M [00:01<00:01, 233MB/s, 9.63MB/s  ] 
model.safetensors: downloading bytes:  53% 264M/499M [00:01<00:00, 330MB/s, 20.8MB/s  ]
model.safetensors: reconstructing file:  67% 335M/499M [00:01<00:00, 281MB/s, 6.54MB/s  ]
model.safetensors: downloading bytes:  63% 315M/499M [00:01<00:00, 301MB/s, 28.6MB/s  ]
model.safetensors: downloading bytes: 100% 335M/335M [00:02<00:00, 163MB/s, 30.9MB/s  ]
model.safetensors: reconstructing file: 100% 499M/499M [00:02<00:00, 242MB/s, 46.6MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 6193.62it/s]
[transforme

In [ ]:
# Merge into one out-of-fold label per article. The accuracy printed here is
# RoBERTa AI-vs-human over the ENTIRE dataset, and should look like a held-out
# number (~0.94-0.96 on GossipCop), NOT the ~0.98 a router scores on its own
# training data - that is the check that the merge really is out-of-fold.
!python funnel_system/build_router_oof.py --run_dir "{PF_RUN}" --folds 0 1 2 3 4 --n_articles 520

router out-of-fold labels: 520 articles -> /content/drive/MyDrive/LIFE/funnel_system/run_politifact_cv/router_oof.jsonl
  RoBERTa AI-vs-human accuracy over the ENTIRE dataset: 0.9135
  confusion [rows=true human/AI, cols=pred]: [[273, 18], [27, 202]]
  human articles sent to the AI branch: 18 (3.46% of all)
  AI articles sent to the human branch: 27 (5.19% of all)
  -> LIFE's training pile: 220 articles (18 of them human); BERT's: 300 (27 of them machine)


In [ ]:
# Human branch: plain BERT trained on everything RoBERTa called HUMAN
# (misrouted machine articles included). Config per MEMORY 7o/7q.
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold0.json" --role human_expert --name PolitiFact++ --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold1.json" --role human_expert --name PolitiFact++ --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold2.json" --role human_expert --name PolitiFact++ --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold3.json" --role human_expert --name PolitiFact++ --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold4.json" --role human_expert --name PolitiFact++ --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl" --skip_if_exists

[PolitiFact++ | human_expert | seed=0] train=205 val=24 test=110 (full shared test) | model=bert-base-uncased label=veracity device=cuda
  pile from router (/content/drive/MyDrive/LIFE/funnel_system/run_politifact_cv/router_oof.jsonl): 19/205 training articles are misroutes from the other branch
config.json: 100% 570/570 [00:00<00:00, 2.83MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 259kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 9.97MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 15.6MB/s]

model.safetensors: downloading bytes:  51% 227M/440M [00:01<00:00, 291MB/s, 19.7MB/s  ]
model.safetensors: reconstructing file:  30% 134M/440M [00:01<00:02, 115MB/s]
model.safetensors: downloading bytes:  88% 389M/440M [00:01<00:00, 265MB/s, 34.3MB/s  ]
model.safetensors: downloading bytes: 100% 415M/415M [00:02<00:00, 191MB/s, 37.6MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:02<00:00, 202MB/s, 40.7MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 5984.86it/

In [ ]:
# Control: plain BERT on the SAME pile LIFE gets (everything called AI),
# so 'does LIFE earn its branch' is apples-to-apples.
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold0.json" --role machine_bert --name PolitiFact++ --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold1.json" --role machine_bert --name PolitiFact++ --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold2.json" --role machine_bert --name PolitiFact++ --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold3.json" --role machine_bert --name PolitiFact++ --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold4.json" --role machine_bert --name PolitiFact++ --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl" --skip_if_exists

[PolitiFact++ | machine_bert | seed=0] train=153 val=18 test=110 (full shared test) | model=bert-base-uncased label=veracity device=cuda
  pile from router (/content/drive/MyDrive/LIFE/funnel_system/run_politifact_cv/router_oof.jsonl): 11/153 training articles are misroutes from the other branch
Loading weights: 100% 199/199 [00:00<00:00, 5758.74it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                 

In [ ]:
# Control: one BERT on all four subsets, no routing at all.
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold0.json" --role monolithic --name PolitiFact++ --out_dir "{PF_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold1.json" --role monolithic --name PolitiFact++ --out_dir "{PF_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold2.json" --role monolithic --name PolitiFact++ --out_dir "{PF_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold3.json" --role monolithic --name PolitiFact++ --out_dir "{PF_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold4.json" --role monolithic --name PolitiFact++ --out_dir "{PF_RUN}" --skip_if_exists

[PolitiFact++ | monolithic | seed=0] train=358 val=42 test=110 (full shared test) | model=bert-base-uncased label=veracity device=cuda
Loading weights: 100% 199/199 [00:00<00:00, 5392.42it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architec

In [ ]:
# LIFE's train/test feature files per fold. --route_by makes the training pile the
# router's AI pile; the printed mix says how many human articles landed in it.
# 'test missing 0/N' everywhere means no evaluation-time fallbacks.
!python funnel_system/assemble_life_data.py --features_dir "{PF_FEATURES}" --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold0.json" --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl"
!python funnel_system/assemble_life_data.py --features_dir "{PF_FEATURES}" --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold1.json" --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl"
!python funnel_system/assemble_life_data.py --features_dir "{PF_FEATURES}" --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold2.json" --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl"
!python funnel_system/assemble_life_data.py --features_dir "{PF_FEATURES}" --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold3.json" --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl"
!python funnel_system/assemble_life_data.py --features_dir "{PF_FEATURES}" --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/fold4.json" --out_dir "{PF_RUN}" --route_by "{PF_RUN}/router_oof.jsonl"

[seed 0] LIFE data assembled: train=153 (router AI pile) test=110
  train pile mix: HF=2 HR=9 MF=67 MR=75 | 11 human articles (7.2%) misrouted into LIFE
  HF: features 97/97 | test missing 0/21
  HR: features 194/194 | test missing 0/40
  MF: features 97/97 | test missing 0/21
  MR: features 132/132 | test missing 0/28
[seed 1] LIFE data assembled: train=154 (router AI pile) test=103
  train pile mix: HF=4 HR=6 MF=69 MR=75 | 10 human articles (6.5%) misrouted into LIFE
  HF: features 97/97 | test missing 0/19
  HR: features 194/194 | test missing 0/39
  MF: features 97/97 | test missing 0/19
  MR: features 132/132 | test missing 0/26
[seed 2] LIFE data assembled: train=160 (router AI pile) test=104
  train pile mix: HF=5 HR=8 MF=69 MR=78 | 13 human articles (8.1%) misrouted into LIFE
  HF: features 97/97 | test missing 0/19
  HR: features 194/194 | test missing 0/39
  MF: features 97/97 | test missing 0/19
  MR: features 132/132 | test missing 0/27
[seed 3] LIFE data assembled: train=1

In [ ]:
# LIFE branch per fold - method untouched (LLaMA2-7B features, sigmoid+BCE head).
# The prints below are over the FULL mixed fold (all four subsets), because LIFE must
# answer for whatever the router sends it; LIFE only owns the AI pile in the funnel.
# evaluate_system.py prints both numbers side by side.
!python LIFE_train/train_bce.py --train_path "{PF_RUN}/life_train_seed0.jsonl" --test_path "{PF_RUN}/life_test_seed0.jsonl" --num_train_epochs 50 --seed 0 --pred_out "{PF_RUN}/life_preds_seed0.jsonl"
!python LIFE_train/train_bce.py --train_path "{PF_RUN}/life_train_seed1.jsonl" --test_path "{PF_RUN}/life_test_seed1.jsonl" --num_train_epochs 50 --seed 1 --pred_out "{PF_RUN}/life_preds_seed1.jsonl"
!python LIFE_train/train_bce.py --train_path "{PF_RUN}/life_train_seed2.jsonl" --test_path "{PF_RUN}/life_test_seed2.jsonl" --num_train_epochs 50 --seed 2 --pred_out "{PF_RUN}/life_preds_seed2.jsonl"
!python LIFE_train/train_bce.py --train_path "{PF_RUN}/life_train_seed3.jsonl" --test_path "{PF_RUN}/life_test_seed3.jsonl" --num_train_epochs 50 --seed 3 --pred_out "{PF_RUN}/life_preds_seed3.jsonl"
!python LIFE_train/train_bce.py --train_path "{PF_RUN}/life_train_seed4.jsonl" --test_path "{PF_RUN}/life_test_seed4.jsonl" --num_train_epochs 50 --seed 4 --pred_out "{PF_RUN}/life_preds_seed4.jsonl"

100% 153/153 [00:00<00:00, 6901.12it/s]
100% 110/110 [00:00<00:00, 8682.72it/s]
seed: 0
--------------------------------BCE head (paper Eq 11-12)--------------------------------
Log INFO: do train...
Epoch:   0% 0/50 [00:00<?, ?it/s]
Iteration:   0% 0/5 [00:00<?, ?it/s]
Iteration:  20% 1/5 [00:00<00:03,  1.00it/s]
Iteration:  40% 2/5 [00:01<00:01,  2.09it/s]
Iteration:  60% 3/5 [00:01<00:00,  3.20it/s]
Iteration:  80% 4/5 [00:01<00:00,  4.24it/s]
Iteration: 100% 5/5 [00:01<00:00,  3.03it/s]
epoch 1: train_loss 0.7002505898475647

Iteration:   0% 0/4 [00:00<?, ?it/s]
Iteration:  25% 1/4 [00:00<00:00,  5.50it/s]
Iteration: 100% 4/4 [00:00<00:00, 12.94it/s]
******** Evalation ********
Accuracy: 61.8
Macro F1 Score: 38.2
Precision/Recall per class (real=0, fake=1): 
61.8/100.0 0.0/0.0
************************************************************************************************************************
Epoch:   2% 1/50 [00:02<02:04,  2.54s/it]
Iteration:   0% 0/5 [00:00<?, ?it/s]
Iteratio

In [ ]:
# The result table. Pooled row = all 520 articles, each predicted by models
# that never trained on it.
!python funnel_system/evaluate_system.py --run_dir "{PF_RUN}" --seeds 0 1 2 3 4 --name PolitiFact++ --cv


--- fold 0 (110 test articles, always-majority baseline 0.6182) ---
  funnel           acc=0.8091 macroF1=0.7926 fake P/R 0.784/0.690  (89/110)
  oracle           acc=0.8364 macroF1=0.8250 fake P/R 0.800/0.762  (92/110)
  two_bert         acc=0.7909 macroF1=0.7729 fake P/R 0.757/0.667  (87/110)
  oracle_two_bert  acc=0.8091 macroF1=0.7948 fake P/R 0.769/0.714  (89/110)
  monolithic       acc=0.8364 macroF1=0.8282 fake P/R 0.773/0.810  (92/110)

--- fold 1 (103 test articles, always-majority baseline 0.6311) ---
  funnel           acc=0.8252 macroF1=0.8024 fake P/R 0.833/0.658  (85/103)
  oracle           acc=0.8252 macroF1=0.8024 fake P/R 0.833/0.658  (85/103)
  two_bert         acc=0.8738 macroF1=0.8603 fake P/R 0.879/0.763  (90/103)
  oracle_two_bert  acc=0.8738 macroF1=0.8603 fake P/R 0.879/0.763  (90/103)
  monolithic       acc=0.8932 macroF1=0.8847 fake P/R 0.865/0.842  (92/103)

--- fold 2 (104 test articles, always-majority baseline 0.6346) ---
  funnel           acc=0.8558 mac

## GossipCop++ - 20,505 articles, 5 folds (the citable result)

~4,100 articles per fold, ~14,750 training articles per round. Budget roughly: routers 80 min,
human BERT 100, machine BERT 70, monolithic 165, LIFE 25 - about 7 hours end to end. Re-running
any cell after a disconnect skips the folds already written.

In [ ]:
!python funnel_system/make_folds.py --data_dir "{GOSSIPCOP_DIR}" --out_dir "{GC_RUN}" --k 5 --seed 7 --name GossipCop++

[GossipCop++] 20505 articles in 12252 id-groups -> 5 folds (assignment seed 7)

fold 0 -> /content/drive/MyDrive/LIFE/funnel_system/run_gossipcop_cv/fold0.json
[GossipCop++] fold 0: articles per subset x split:
  HF: train=  2926 val=   341 test=   817
  HR: train=  5895 val=   639 test=  1634
  MF: train=  2926 val=   341 test=   817
  MR: train=  3001 val=   334 test=   834
  test: 4102 articles | fake 1634 (39.8%) | AI 1651 (40.2%)

fold 1 -> /content/drive/MyDrive/LIFE/funnel_system/run_gossipcop_cv/fold1.json
[GossipCop++] fold 1: articles per subset x split:
  HF: train=  2923 val=   344 test=   817
  HR: train=  5898 val=   636 test=  1634
  MF: train=  2923 val=   344 test=   817
  MR: train=  3005 val=   330 test=   834
  test: 4102 articles | fake 1634 (39.8%) | AI 1651 (40.2%)

fold 2 -> /content/drive/MyDrive/LIFE/funnel_system/run_gossipcop_cv/fold2.json
[GossipCop++] fold 2: articles per subset x split:
  HF: train=  2945 val=   322 test=   817
  HR: train=  5876 val=   6

In [ ]:
# Routers first: fold k's router predicts fold k, which is what makes the merged
# labels out-of-fold. Nothing downstream can run until all 5 are written.
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold0.json" --role router --name GossipCop++ --out_dir "{GC_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold1.json" --role router --name GossipCop++ --out_dir "{GC_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold2.json" --role router --name GossipCop++ --out_dir "{GC_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold3.json" --role router --name GossipCop++ --out_dir "{GC_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold4.json" --role router --name GossipCop++ --out_dir "{GC_RUN}" --skip_if_exists

[GossipCop++ | router | seed=0] train=14007 val=1655 test=4102 (full shared test) | model=roberta-base label=provenance device=cuda
Loading weights: 100% 197/197 [00:00<00:00, 5047.45it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
  epoch 1: 

In [ ]:
# Merge into one out-of-fold label per article. The accuracy printed here is
# RoBERTa AI-vs-human over the ENTIRE dataset, and should look like a held-out
# number (~0.94-0.96 on GossipCop), NOT the ~0.98 a router scores on its own
# training data - that is the check that the merge really is out-of-fold.
!python funnel_system/build_router_oof.py --run_dir "{GC_RUN}" --folds 0 1 2 3 4 --n_articles 20505

router out-of-fold labels: 20505 articles -> /content/drive/MyDrive/LIFE/funnel_system/run_gossipcop_cv/router_oof.jsonl
  RoBERTa AI-vs-human accuracy over the ENTIRE dataset: 0.9501
  confusion [rows=true human/AI, cols=pred]: [[11541, 711], [313, 7940]]
  human articles sent to the AI branch: 711 (3.47% of all)
  AI articles sent to the human branch: 313 (1.53% of all)
  -> LIFE's training pile: 8651 articles (711 of them human); BERT's: 11854 (313 of them machine)


In [ ]:
# Human branch: plain BERT trained on everything RoBERTa called HUMAN
# (misrouted machine articles included). Config per MEMORY 7o/7q.
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold0.json" --role human_expert --name GossipCop++ --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold1.json" --role human_expert --name GossipCop++ --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold2.json" --role human_expert --name GossipCop++ --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold3.json" --role human_expert --name GossipCop++ --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold4.json" --role human_expert --name GossipCop++ --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl" --skip_if_exists

[GossipCop++ | human_expert | seed=0] train=7920 val=949 test=4102 (full shared test) | model=bert-base-uncased label=veracity device=cuda
  pile from router (/content/drive/MyDrive/LIFE/funnel_system/run_gossipcop_cv/router_oof.jsonl): 230/7920 training articles are misroutes from the other branch
Loading weights: 100% 199/199 [00:00<00:00, 5769.29it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight              

In [ ]:
# Control: plain BERT on the SAME pile LIFE gets (everything called AI),
# so 'does LIFE earn its branch' is apples-to-apples.
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold0.json" --role machine_bert --name GossipCop++ --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold1.json" --role machine_bert --name GossipCop++ --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold2.json" --role machine_bert --name GossipCop++ --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold3.json" --role machine_bert --name GossipCop++ --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold4.json" --role machine_bert --name GossipCop++ --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl" --skip_if_exists

[GossipCop++ | machine_bert | seed=0] train=6107 val=706 test=4102 (full shared test) | model=bert-base-uncased label=veracity device=cuda
  pile from router (/content/drive/MyDrive/LIFE/funnel_system/run_gossipcop_cv/router_oof.jsonl): 421/6107 training articles are misroutes from the other branch
Loading weights: 100% 199/199 [00:00<00:00, 5677.35it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight              

In [ ]:
# Control: one BERT on all four subsets, no routing at all.
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold0.json" --role monolithic --name GossipCop++ --out_dir "{GC_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold1.json" --role monolithic --name GossipCop++ --out_dir "{GC_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold2.json" --role monolithic --name GossipCop++ --out_dir "{GC_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold3.json" --role monolithic --name GossipCop++ --out_dir "{GC_RUN}" --skip_if_exists
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold4.json" --role monolithic --name GossipCop++ --out_dir "{GC_RUN}" --skip_if_exists

[GossipCop++ | monolithic | seed=0] train=14007 val=1655 test=4102 (full shared test) | model=bert-base-uncased label=veracity device=cuda
Loading weights: 100% 199/199 [00:00<00:00, 4630.50it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

In [ ]:
# LIFE's train/test feature files per fold. --route_by makes the training pile the
# router's AI pile; the printed mix says how many human articles landed in it.
# 'test missing 0/N' everywhere means no evaluation-time fallbacks.
!python funnel_system/assemble_life_data.py --features_dir "{GC_FEATURES}" --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold0.json" --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl"
!python funnel_system/assemble_life_data.py --features_dir "{GC_FEATURES}" --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold1.json" --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl"
!python funnel_system/assemble_life_data.py --features_dir "{GC_FEATURES}" --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold2.json" --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl"
!python funnel_system/assemble_life_data.py --features_dir "{GC_FEATURES}" --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold3.json" --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl"
!python funnel_system/assemble_life_data.py --features_dir "{GC_FEATURES}" --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/fold4.json" --out_dir "{GC_RUN}" --route_by "{GC_RUN}/router_oof.jsonl"

[seed 0] LIFE data assembled: train=6118 (router AI pile) test=4102
  train pile mix: HF=153 HR=275 MF=2923 MR=2767 | 428 human articles (7.0%) misrouted into LIFE
  HF: features 4084/4084 | test missing 0/817
  HR: features 8168/8168 | test missing 0/1634
  MF: features 4084/4084 | test missing 0/817
  MR: features 4169/4169 | test missing 0/834
[seed 1] LIFE data assembled: train=6287 (router AI pile) test=4102
  train pile mix: HF=200 HR=366 MF=2919 MR=2802 | 566 human articles (9.0%) misrouted into LIFE
  HF: features 4084/4084 | test missing 0/817
  HR: features 8168/8168 | test missing 0/1634
  MF: features 4084/4084 | test missing 0/817
  MR: features 4169/4169 | test missing 0/834
[seed 2] LIFE data assembled: train=6103 (router AI pile) test=4102
  train pile mix: HF=120 HR=259 MF=2943 MR=2781 | 379 human articles (6.2%) misrouted into LIFE
  HF: features 4084/4084 | test missing 0/817
  HR: features 8168/8168 | test missing 0/1634
  MF: features 4084/4084 | test missing 0/817

In [ ]:
# LIFE branch per fold - method untouched (LLaMA2-7B features, sigmoid+BCE head).
# The prints below are over the FULL mixed fold (all four subsets), because LIFE must
# answer for whatever the router sends it; LIFE only owns the AI pile in the funnel.
# evaluate_system.py prints both numbers side by side.
!python LIFE_train/train_bce.py --train_path "{GC_RUN}/life_train_seed0.jsonl" --test_path "{GC_RUN}/life_test_seed0.jsonl" --num_train_epochs 8 --seed 0 --pred_out "{GC_RUN}/life_preds_seed0.jsonl"
!python LIFE_train/train_bce.py --train_path "{GC_RUN}/life_train_seed1.jsonl" --test_path "{GC_RUN}/life_test_seed1.jsonl" --num_train_epochs 8 --seed 1 --pred_out "{GC_RUN}/life_preds_seed1.jsonl"
!python LIFE_train/train_bce.py --train_path "{GC_RUN}/life_train_seed2.jsonl" --test_path "{GC_RUN}/life_test_seed2.jsonl" --num_train_epochs 8 --seed 2 --pred_out "{GC_RUN}/life_preds_seed2.jsonl"
!python LIFE_train/train_bce.py --train_path "{GC_RUN}/life_train_seed3.jsonl" --test_path "{GC_RUN}/life_test_seed3.jsonl" --num_train_epochs 8 --seed 3 --pred_out "{GC_RUN}/life_preds_seed3.jsonl"
!python LIFE_train/train_bce.py --train_path "{GC_RUN}/life_train_seed4.jsonl" --test_path "{GC_RUN}/life_test_seed4.jsonl" --num_train_epochs 8 --seed 4 --pred_out "{GC_RUN}/life_preds_seed4.jsonl"

Streaming output truncated to the last 5000 lines.
Iteration:  87% 167/191 [00:18<00:02,  9.49it/s]
Iteration:  88% 168/191 [00:18<00:02,  9.42it/s]
Iteration:  88% 169/191 [00:18<00:02,  9.40it/s]
Iteration:  89% 170/191 [00:18<00:02,  9.40it/s]
Iteration:  90% 171/191 [00:18<00:02,  9.49it/s]
Iteration:  90% 172/191 [00:18<00:01,  9.54it/s]
Iteration:  91% 173/191 [00:18<00:01,  9.58it/s]
Iteration:  91% 174/191 [00:18<00:01,  9.54it/s]
Iteration:  92% 175/191 [00:18<00:01,  9.51it/s]
Iteration:  92% 176/191 [00:19<00:01,  9.48it/s]
Iteration:  93% 177/191 [00:19<00:01,  9.54it/s]
Iteration:  93% 178/191 [00:19<00:01,  9.56it/s]
Iteration:  94% 179/191 [00:19<00:01,  9.54it/s]
Iteration:  94% 180/191 [00:19<00:01,  9.52it/s]
Iteration:  95% 181/191 [00:19<00:01,  9.44it/s]
Iteration:  95% 182/191 [00:19<00:00,  9.51it/s]
Iteration:  96% 183/191 [00:19<00:00,  9.42it/s]
Iteration:  96% 184/191 [00:19<00:00,  9.40it/s]
Iteration:  97% 185/191 [00:19<00:00,  9.42it/s]
Iteration:  97% 18

In [ ]:
# The result table. Pooled row = all 20505 articles, each predicted by models
# that never trained on it.
!python funnel_system/evaluate_system.py --run_dir "{GC_RUN}" --seeds 0 1 2 3 4 --name GossipCop++ --cv


--- fold 0 (4102 test articles, always-majority baseline 0.6017) ---
  funnel           acc=0.8515 macroF1=0.8424 fake P/R 0.847/0.766  (3493/4102)
  oracle           acc=0.8654 macroF1=0.8579 fake P/R 0.855/0.797  (3550/4102)
  two_bert         acc=0.8774 macroF1=0.8715 fake P/R 0.856/0.832  (3599/4102)
  oracle_two_bert  acc=0.8818 macroF1=0.8759 fake P/R 0.864/0.834  (3617/4102)
  monolithic       acc=0.8784 macroF1=0.8718 fake P/R 0.869/0.818  (3603/4102)

--- fold 1 (4102 test articles, always-majority baseline 0.6017) ---
  funnel           acc=0.8462 macroF1=0.8397 fake P/R 0.806/0.809  (3471/4102)
  oracle           acc=0.8528 macroF1=0.8467 fake P/R 0.811/0.821  (3498/4102)
  two_bert         acc=0.8708 macroF1=0.8661 fake P/R 0.825/0.857  (3572/4102)
  oracle_two_bert  acc=0.8723 macroF1=0.8675 fake P/R 0.828/0.858  (3578/4102)
  monolithic       acc=0.8679 macroF1=0.8632 fake P/R 0.819/0.858  (3560/4102)

--- fold 2 (4102 test articles, always-majority baseline 0.6017) ---


## Notes / caveats (read before quoting numbers)

- **Pooled = the whole dataset.** Each article sits in exactly one test fold, so the pooled row
  covers all 20,505 / 520 articles with no article graded by a model that trained on it. The
  per-fold spread is the variance estimate (it replaces the 3-seed spread of the held-out run).
- **Compare against `funnel_colab.ipynb` (MEMORY 7s)**, which used one 70/15/15 split and
  true-subset training: GossipCop funnel 0.8580, two_bert 0.8787, monolithic 0.8696. Differences
  here come from two changes at once - full coverage and router-filtered training.
- **Feature-space leak (inherited, unfixed):** LIFE's key-sentence extractor and LLaMA features
  were generated once over all articles, so its feature space saw every article; only the trained
  classifiers respect the folds. A clean rerun regenerates features per fold (hours per fold).
- **Second-order leak (new, noted):** fold k's experts train on articles whose router labels came
  from other folds' routers, which did see fold k. The article being predicted is never seen by
  its own predictors; removing this entirely needs nested CV (25 router trainings).
- **Veracity convention:** fake=1 / real=0 everywhere in funnel_system (matches LIFE's BCE head).
- `bce_en.pt` is clobbered by each LIFE run, so keep the LIFE cell's lines sequential.